### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="housing_prices_metropolitan_india",
    dataset_year="2020",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/ruchi798/housing-prices-in-metropolitan-areas-of-india",
    download_description="""
We download the data from Kaggle.

kaggle datasets download ruchi798/housing-prices-in-metropolitan-areas-of-india && unzip housing-prices-in-metropolitan-areas-of-india.zip && rm housing-prices-in-metropolitan-areas-of-india.zip && mkdir -p local-data-warehouse/housing_prices_metropolitan_india && mv *.csv local-data-warehouse/housing_prices_metropolitan_india/
""",
    # References
    academic_reference_bibtex=r"""@misc{Bhatia2019HousingPricesInMetropolitanAreasOfIndia,
  author = {Ruchi Bhatia},
  title  = {Housing Prices in Metropolitan Areas of India},
  year   = {2019},
  howpublished = {\url{https://www.kaggle.com/datasets/ruchi798/housing-prices-in-metropolitan-areas-of-india}},
  note   = {Kaggle dataset}
}
""",
    academic_reference_bibtex_key="Bhatia2019HousingPricesInMetropolitanAreasOfIndia",
    license="CC0: Public Domain",
    data_tags=["IID", "Spatial"],
    curation_comments="""
We start with all .csv files from Kaggle and merge them, following the notebook from data creator (https://www.kaggle.com/code/ruchi798/housing-prices-eda-and-prediction).
We treat the task as an IID task, since we predict the prices of houses from the past/present, not the future. In other words, we aim to "fill gaps" in the house price prediction task, following the work from the data creator. This can also be seen as a benchmarking how good a model can extract signal from the data for the sake of feature importance as a scientific discovery task.

- We follow other house price prediction tasks and normalize the price by area and then log scale it. We keep total area as a feature.
- We replace 9s with nans ("nothing was mentioned about certain amenities, '9' was used to mark such values")
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="LogPricePerArea",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

dfs = []
for f in ["Bangalore.csv", "Chennai.csv", "Delhi.csv", "Hyderabad.csv", "Kolkata.csv", "Mumbai.csv"]:  # TODO: verify filenames after download
    dfs.append(pd.read_csv(dataset_mold.path / f))
    dfs[-1]["City"] = f.split(".")[0]  # Add city as a feature
df = pd.concat(dfs, ignore_index=True)
print("Loaded data shape:", df.shape)

Loaded data shape: (32963, 41)


In [4]:
org_df = df.copy()

In [3]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,Price,Area,Location,No. of Bedrooms,Resale,MaintenanceStaff,Gymnasium,SwimmingPool,LandscapedGardens,JoggingTrack,RainWaterHarvesting,IndoorGames,ShoppingMall,Intercom,SportsFacility,ATM,ClubHouse,School,24X7Security,PowerBackup,CarParking,StaffQuarter,Cafeteria,MultipurposeRoom,Hospital,WashingMachine,Gasconnection,AC,Wifi,Children'splayarea,LiftAvailable,BED,VaastuCompliant,Microwave,GolfCourse,TV,DiningTable,Sofa,Wardrobe,Refrigerator,City
0,30000000,3340,JP Nagar Phase 1,4,0,1,1,1,1,1,1,1,0,1,1,0,1,0,1,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,Bangalore
1,7888000,1045,Dasarahalli on Tumkur Road,2,0,0,1,1,1,1,1,1,0,0,1,0,1,0,1,1,1,0,0,1,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,Bangalore
2,4866000,1179,Kannur on Thanisandra Main Road,2,0,0,1,1,1,1,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,Bangalore
3,8358000,1675,Doddanekundi,3,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,Bangalore
4,6845000,1670,Kengeri,3,0,1,1,1,1,1,1,1,0,1,1,0,1,0,1,1,1,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,Bangalore


In [7]:
df.duplicated(subset=["Area", "Location", "No. of Bedrooms", "City"]).sum() # -> temporal duplicates, cannot be used without temporal information.

np.int64(15103)

In [5]:
df = org_df.copy()

# Normalize price by area and log scale it
df["PricePerArea"] = df["Price"] / df["Area"]
df["LogPricePerArea"] = np.log(df["PricePerArea"])
df = df.drop(columns=["Price", "PricePerArea"])

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [ ]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)